# 01. Exploratory Data Analysis & Data Preprocessing
Sistem Segmentasi Jenis Usaha UMKM DKI Jakarta Berbasis Wilayah Menggunakan Algoritma K-Modes Clustering  

---

### **Tujuan Notebook:**
1. Memuat dataset mentah UMKM DKI Jakarta (`.xls` berformat tabel HTML).
2. Melakukan audit kualitas data (*null values*, tipe data, dan ukuran data).
3. Melakukan prapemrosesan dan pembersihan baris data yang cacat/kosong pada atribut krusial (`jenis_usaha`).
4. Menyimpan dataset bersih ke `data/processed/01_umkm_clean.csv`.

In [2]:
import os
import pandas as pd
import numpy as np

# Pengaturan opsi tampilan Pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

print("Pustaka sukses dimuat.")

Pustaka sukses dimuat.


## 1. Input Dataset Mentah
Karena file `.xls` tersebut terstruktur dalam format tabel HTML, pembacaan dilakukan menggunakan fungsi `pd.read_html()`.

In [3]:
# Path lokasi file mentah
raw_data_path = os.path.join('..', 'data', 'raw', 'Data_Usaha Mikro, Kecil, dan Menengah (UMKM) - tabel.xls')

# Jika notebook dijalankan langsung di direktori utama (root), sesuaikan path:
if not os.path.exists(raw_data_path):
    raw_data_path = 'Data_Usaha Mikro, Kecil, dan Menengah (UMKM) - tabel.xls'

print(f"Membaca berkas dari: {raw_data_path}")
tables = pd.read_html(raw_data_path)
df_raw = tables[0]

print(f"Dimensi Data Mentah (Baris, Kolom): {df_raw.shape}")
df_raw.head(5)

Membaca berkas dari: ..\data\raw\Data_Usaha Mikro, Kecil, dan Menengah (UMKM) - tabel.xls
Dimensi Data Mentah (Baris, Kolom): (298893, 7)


,periode_data,wilayah,skpd,jenis_usaha,nama_usaha,tahun,nama_pengusaha
0,2024,KOTA ADM. JAKARTA PUSAT,DINAS PARIWISATA DAN EKONOMI KREATIF,SABLON DAN DESIGN GRAFIS,RADH DESIGN,2020,RADHIYYAH MARDHIYYAH
1,2024,KOTA ADM. JAKARTA SELATAN,DINAS PARIWISATA DAN EKONOMI KREATIF,MAKANAN DAN MINUMAN,RADIMSUM,2021,LIDIA NASTALIA
2,2024,KAB. ADM. KEP. SERIBU,DINAS PARIWISATA DAN EKONOMI KREATIF,AGEN WISATA PERJALANAN (AWP),RADWA WISATA,2020,EVA OKTAVIANI
3,2024,KOTA ADM. JAKARTA PUSAT,DINAS PARIWISATA DAN EKONOMI KREATIF,MAKANAN DAN MINUMAN,RAFARIFOOD_IKANSEGAR,2020,CARIDI
4,2024,KOTA ADM. JAKARTA TIMUR,DINAS PARIWISATA DAN EKONOMI KREATIF,MAKANAN DAN MINUMAN,RAFIANTUN UDUK,2022,RAFIATUN


## 2. Audit Kualitas Data
Langkah ini bertujuan untuk mengidentifikasi keberadaan nilai hilang (*missing values*) serta kesesuaian tipe data pada setiap atribut.

In [4]:
audit_summary = pd.DataFrame({
    'Tipe Data': df_raw.dtypes,
    'Missing Value': df_raw.isnull().sum(),
    'Persentase Missing (%)': (df_raw.isnull().sum() / len(df_raw)) * 100,
    'Kategori Unik': df_raw.nunique()
})

print("=== RINGKASAN AUDIT DATA MENTAH ===")
audit_summary

=== RINGKASAN AUDIT DATA MENTAH ===


,Tipe Data,Missing Value,Persentase Missing (%),Kategori Unik
periode_data,int64,0,0.000000,2
wilayah,str,0,0.000000,6
skpd,str,0,0.000000,49
jenis_usaha,str,18,0.006022,161
nama_usaha,str,183,0.061226,228496
tahun,int64,0,0.000000,6
nama_pengusaha,str,0,0.000000,176619


## 3. Pembersihan Data (*Data Cleaning*)
Sesuai rancangan metodologi, atribut `jenis_usaha` merupakan variabel kategorikal utama. Baris data yang memiliki nilai `NaN` pada kolom `jenis_usaha` akan dieliminasi[cite: 1].

In [8]:
df_clean = df_raw.dropna(subset=['jenis_usaha']).copy()

print("=== HASIL PEMBERSIHAN DATA ===")
print(f"Total baris mentah     : {len(df_raw):,} baris")
print(f"Baris tereliminasi     : {len(df_raw) - len(df_clean):,} baris")
print(f"Total baris data bersih: {len(df_clean):,} baris")

# Konversi eksplisit seluruh kolom kategorikal menjadi tipe string
for col in ['wilayah', 'skpd', 'jenis_usaha']:
    df_clean[col] = df_clean[col].astype(str).str.strip()

=== HASIL PEMBERSIHAN DATA ===
Total baris mentah     : 298,893 baris
Baris tereliminasi     : 18 baris
Total baris data bersih: 298,875 baris


## 4. Eksplorasi Awal Distribusi Wilayah Administrasi
Mengecek distribusi populasi UMKM pada 6 wilayah administrasi di DKI Jakarta.

In [11]:
print("=== DISTRIBUSI UMKM PER WILAYAH ADMINISTRASI ===")
display(df_clean['wilayah'].value_counts().reset_index())

print("\n=== 10 JENIS USAHA TERBANYAK DI DKI JAKARTA ===")
display(df_clean['jenis_usaha'].value_counts().head(10).reset_index())

=== DISTRIBUSI UMKM PER WILAYAH ADMINISTRASI ===


,wilayah,count
0,KOTA ADM. JAKARTA SELATAN,83890
1,KOTA ADM. JAKARTA TIMUR,63004
2,KOTA ADM. JAKARTA BARAT,59089
3,KOTA ADM. JAKARTA UTARA,47942
4,KOTA ADM. JAKARTA PUSAT,40956
5,KAB. ADM. KEP. SERIBU,3994



=== 10 JENIS USAHA TERBANYAK DI DKI JAKARTA ===


,jenis_usaha,count
0,MAKANAN DAN MINUMAN,135295
1,PRODUK MAKANAN LAINNYA,40484
2,FASHION,17909
3,WARUNG SEMBAKO,17605
4,KUE BASAH,8076
5,KEDAI MAKANAN,6810
6,KERAJINAN / SOUVENIR,6440
7,RUMAH/WARUNG MAKAN,5339
8,"WARUNG KOPI / KEDAI KOPI , KEDAI JUS DAN MINUMAN LAINNYA",4742
9,OLAHAN PERTANIAN,3862


## 5. Penyimpanan Dataset Bersih Tahap 1
Menyimpan dataset yang telah dibersihkan ke direktori `data/processed/` sebagai acuan untuk tahap rekayasa fitur.

In [13]:
output_dir = os.path.join('..', 'data', 'processed')
os.makedirs(output_dir, exist_ok=True)

clean_output_path = os.path.join(output_dir, '01_umkm_clean.csv')
df_clean.to_csv(clean_output_path, index=False)

print(f"Dataset bersih berhasil disimpan di: {clean_output_path}")

Dataset bersih berhasil disimpan di: ..\data\processed\01_umkm_clean.csv
